In [ ]:
from fedbiomed.common.training_plans import FedSGDClassifier
import pandas as pd
from fedbiomed.common.dataset import NipoppyDataset

class SGDClassifierTrainingPlan(FedSGDClassifier):
    def init_dependencies(self):
        deps = [
                "from skrub import TableVectorizer",
                "from sklearn.preprocessing import OneHotEncoder",
                "import pandas as pd",
                "from fedbiomed.common.dataset import NipoppyDataset",
               ]
        return deps

    @staticmethod
    def transform_skrub(df: pd.DataFrame) -> pd.DataFrame:
        specific_transformers = []
        if NipoppyDataset.TERMURL_SEX in df.columns:
            specific_transformers.append(
                (
                    OneHotEncoder(
                        drop=[NipoppyDataset.TERMURL_MALE], sparse_output=False
                    ),
                    [NipoppyDataset.TERMURL_SEX],
                )
            )
        if NipoppyDataset.TERMURL_COG_DECLINE in df.columns:
            specific_transformers.append(
                (
                    OneHotEncoder(
                        drop=[NipoppyDataset.TERMURL_UNAVAILABLE],
                        sparse_output=False,
                        feature_name_combiner=lambda x, _: x,
                    ),
                    [NipoppyDataset.TERMURL_COG_DECLINE],
                )
            )

        table_vectorizer = TableVectorizer(specific_transformers=specific_transformers)
        df = table_vectorizer.fit_transform(df)
        return df
    
    def training_data(self):
        dataset = NipoppyDataset(phenotypes=[
                                    NipoppyDataset.TERMURL_AGE,
                                    NipoppyDataset.TERMURL_SEX,
                                    NipoppyDataset.TERMURL_DIAGNOSIS,
                                    NipoppyDataset.TERMURL_COG_DECLINE_AVAILABILITY,
                                ],
                                derivatives=[("freesurfer",
                                              "7.3.2",
                                              "idp/fs_stats-0.2.1/fs7.3.2-aparc.DKTatlas-thickness.tsv",)],
                                target=[NipoppyDataset.TERMURL_COG_DECLINE],
                                session_filters='01',  # session filter
                                drop_na=True,
                                sample_level_transform=None,
                                sample_level_target_transform=None,
                                whole_df_level_transform=SGDClassifierTrainingPlan.transform_skrub)
        return DataManager(dataset=dataset, shuffle=True)



In [ ]:
# harcoded for now... how to get a better idea in the future?
n_features = 71

2026-04-08 11:19:53,381 fedbiomed WARNING - Node NODE_1a60d3f8-fb8c-4085-95de-fbc75e165b20 is disconnected. Request/task that are created for this node will be flushed

In [9]:
model_args = {
            # fedbiomed
            "eta0": 0.05,
            "random_state": 424242,
            "n_features": n_features,
            "n_classes": 2,
            # model
            "learning_rate": "invscaling",
            "penalty": "l2",
        }

In [10]:
training_args = {
    'num_updates': 5,  # 50,
    "loader_args": {"batch_size": 50}
}

2026-04-08 11:18:02,069 fedbiomed INFO - Node NODE_1a60d3f8-fb8c-4085-95de-fbc75e165b20 is back online!

In [11]:
from fedbiomed.researcher.federated_workflows import Experiment
from fedbiomed.researcher.aggregators.fedavg import FedAverage

tags =  ['nipoppy']
rounds = 2

# search for corresponding datasets across nodes datasets
exp = Experiment(tags=tags,
                 model_args=model_args,
                 training_plan_class=SGDClassifierTrainingPlan,
                 training_args=training_args,
                 round_limit=rounds,
                 aggregator=FedAverage(),
                 node_selection_strategy=None)

2026-04-08 11:18:02,242 fedbiomed INFO - Updating training data. This action will update FederatedDataset, and the nodes that will participate to the experiment.

2026-04-08 11:18:02,257 fedbiomed INFO - Node selected for training -> Default Node Name
Node ID is -> NODE_1a60d3f8-fb8c-4085-95de-fbc75e165b20

2026-04-08 11:18:02,267 fedbiomed WARNING - Option share_persistent_buffers is not supported in SKLearnTrainingPlan, it will be ignored.

In [12]:
exp.run()

2026-04-08 11:18:02,440 fedbiomed INFO - Sampled nodes in round 0 ['NODE_1a60d3f8-fb8c-4085-95de-fbc75e165b20']

2026-04-08 11:18:02,443 fedbiomed INFO - Sending request 
					 To: NODE_1a60d3f8-fb8c-4085-95de-fbc75e165b20 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-04-08 11:18:03,602 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_1a60d3f8-fb8c-4085-95de-fbc75e165b20 
					 Node Name: Default Node Name 
					 Round 1 | Iteration: 1/5 (20%) | Samples: 1/250
 					 Loss hinge: 1.000000 
					 ---------

2026-04-08 11:18:03,603 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_1a60d3f8-fb8c-4085-95de-fbc75e165b20 
					 Node Name: Default Node Name 
					 Round 1 | Iteration: 2/5 (40%) | Samples: 2/250
 					 Loss hinge: 0.000000 
					 ---------

2026-04-08 11:18:03,604 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_1a60d3f8-fb8c-4085-95de-fbc75e165b20 
					 Node Name: Default Node Name 
					 Round 1 | Iteration: 3/5 (60%) | Samples: 3/250
 					 Loss hinge: 0.000000 
					 ---------

...

2026-04-08 11:18:03,859 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_1a60d3f8-fb8c-4085-95de-fbc75e165b20 
					 Node Name: Default Node Name 
					 Round 2 | Iteration: 3/5 (60%) | Samples: 3/250
 					 Loss hinge: 0.000000 
					 ---------

2026-04-08 11:18:03,860 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_1a60d3f8-fb8c-4085-95de-fbc75e165b20 
					 Node Name: Default Node Name 
					 Round 2 | Iteration: 4/5 (80%) | Samples: 4/250
 					 Loss hinge: 0.000000 
					 ---------

2026-04-08 11:18:03,861 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_1a60d3f8-fb8c-4085-95de-fbc75e165b20 
					 Node Name: Default Node Name 
					 Round 2 | Iteration: 5/5 (100%) | Samples: 5/250
 					 Loss hinge: 0.000000 
					 ---------

2026-04-08 11:18:03,870 fedbiomed INFO - Nodes that successfully reply in round 1 ['NODE_1a60d3f8-fb8c-4085-95de-fbc75e165b20']

2

In [15]:
import os
os.getcwd()

'/Users/fcremone/dev/projects/nipoppy/nipoppy-researcher/notebooks'